# NEB Input Generation — Edge Dislocation ⟨111⟩{110} in Fe

This notebook generates, minimizes, and aligns the **initial** and **final** endpoint configurations  
for a Nudged Elastic Band (NEB) calculation of edge dislocation glide in BCC iron.

**Pipeline:**
1. Generate dislocation structures with Atomsk  
2. Assign mobile / fixed atom types  
3. Two-stage LAMMPS minimization (FIRE → CG)  
4. Re-align via KDTree + MIC unwrapping  
5. Export NEB-ready `.config` files  
6. Format final endpoint for LAMMPS NEB  
7. Assemble case directory & write LAMMPS NEB input script

## 1 · Environment Setup & Directory Initialisation

In [7]:
import os
import numpy as np
from pathlib import Path
import shutil, subprocess
import matplotlib.pyplot as plt

# Simulation & Geometry Tools
from lammps import lammps
from ase import Atoms
from ase.io import read, write
from ase.geometry import find_mic
from scipy.spatial import cKDTree

# --- Directory Management ---
base_dir = Path.cwd().parent
input_dir = base_dir / "input_raw"
output_dir = base_dir / "input"

# Clear existing input directory to prevent file conflicts
if input_dir.exists():
    shutil.rmtree(input_dir)

# Recreate fresh directories
input_dir.mkdir(parents=True, exist_ok=True)
output_dir.mkdir(parents=True, exist_ok=True)

potential_file = base_dir / "potentials/mendelev03.fs"
print(f"Directory cleared. Working in: {input_dir}")
print(f"Potential:  {potential_file}")
print(f"Output dir: {output_dir}")

Directory cleared. Working in: /home/Ethan/Projects/neb_2/input_raw
Potential:  /home/Ethan/Projects/neb_2/potentials/mendelev03.fs
Output dir: /home/Ethan/Projects/neb_2/input


## 2 · Generate Dislocation Structures (Atomsk)

Creates two BCC Fe supercells with an edge dislocation inserted at:
- **Initial**: `X_INI`  
- **Final**: `X_INI + Burgers vector` (one glide step)

Crystal orientation: **[111] || X**, **[1-10] || Y**, **[11-2] || Z**

In [8]:
# --- Configuration ---
LAT_PARAM = 2.8665
X_SIZE = 120
Y_SIZE = 21
Z_SIZE = 3

# --- Execution ---
print(f"Generating 110 structures in: {input_dir}")

unitcell_path = input_dir / f"unitcell.lmp"
topslab_path = input_dir / f"topslab.lmp"
botslab_path = input_dir / f"botslab.lmp"
final_output = input_dir / f"Fe_E111_112_raw.lmp"

final_output.unlink(missing_ok=True)

top_deform = -0.5 / (X_SIZE + 1)
bot_deform = 0.5 / X_SIZE

subprocess.run(["atomsk", "--create", "bcc", str(LAT_PARAM), "Fe", 
                "orient", "[111]", "[11-2]", "[1-10]", str(unitcell_path)], check=True, capture_output=True)

subprocess.run(["atomsk", str(unitcell_path), "-duplicate", str(X_SIZE + 1), str(Y_SIZE), str(Z_SIZE),
                "-deform", "x", f"{top_deform:.6f}", "0.0", str(topslab_path)], check=True, capture_output=True)

subprocess.run(["atomsk", str(unitcell_path), "-duplicate", str(X_SIZE), str(Y_SIZE), str(Z_SIZE),
                "-deform", "x", f"{bot_deform:.6f}", "0.0", str(botslab_path)], check=True, capture_output=True)

subprocess.run(["atomsk", "--merge", "stack", "y", "2", str(botslab_path), str(topslab_path), str(final_output)], check=True, capture_output=True)

for tmp_file in [unitcell_path, topslab_path, botslab_path]:
    tmp_file.unlink(missing_ok=True)

print("\n110 configurations generated.")

Generating 110 structures in: /home/Ethan/Projects/neb_2/input_raw

110 configurations generated.


## 3 · Assign Atom Types (Mobile / Fixed)

Atoms in the top and bottom `NUM_FIXED_LAYERS` layers are assigned **Type 2 ("Ru")** and  
will be frozen during minimization. All others are **Type 1 ("Fe")** and are free to relax.

> The "Ru" label is a LAMMPS grouping convention — there is no actual ruthenium.

In [9]:
TOTAL_Y_LAYERS  = int(Y_SIZE) * 2   # Layers in Y direction
NUM_FIXED_LAYERS = 6

atoms = read(str(final_output), format="lammps-data", atom_style="atomic")

# 1. Sort atoms: X → Y → Z for consistency
pos      = atoms.get_positions()
sort_idx = np.lexsort((pos[:, 2], pos[:, 1], pos[:, 0]))
atoms    = atoms[sort_idx].copy()
atoms.center()

# 2. Identify fixed boundary layers (Split into Top and Bottom)
y_coords = atoms.get_positions()[:, 1]
y_min, y_max = y_coords.min(), y_coords.max()
y_height = y_max - y_min

layer_thickness    = y_height / TOTAL_Y_LAYERS
boundary_threshold = layer_thickness * (NUM_FIXED_LAYERS / 2) # Adjust if needed

# Define masks for top and bottom specifically
bottom_mask = (y_coords <= y_min + boundary_threshold + 0.01)
top_mask    = (y_coords >= y_max - boundary_threshold - 0.01)

# 3. Assign types
# Default is Type 1 (Mobile)
new_types = np.ones(len(atoms), dtype=int)

# Assign Type 2 to Top, Type 3 to Bottom
new_types[top_mask] = 2
new_types[bottom_mask] = 3

atoms.set_array('type', new_types)

# Update chemical symbols for visualization (Fe, Ru, Ni or similar)
# Type 1: Fe, Type 2: Ru (Top), Type 3: Ni (Bottom)
sym_map = {1: 'Fe', 2: 'Ru', 3: 'Ni'}
atoms.set_chemical_symbols([sym_map[t] for t in new_types])

atoms.set_pbc([True, False, True])

# 4. Write
out_path = output_dir / f"typed_{final_output.name}"
# Note: specorder must match the number of types (1, 2, 3)
write(str(out_path), atoms, format="lammps-data",
        specorder=["Fe", "Fe", "Fe"], atom_style="atomic")

n_mobile = int(np.sum(new_types == 1))
n_top    = int(np.sum(new_types == 2))
n_bot    = int(np.sum(new_types == 3))

print(f"{out_path.name}:")
print(f"  - {n_mobile} Mobile (Type 1)")
print(f"  - {n_top} Top Fixed (Type 2)")
print(f"  - {n_bot} Bottom Fixed (Type 3)")

typed_Fe_E111_112_raw.lmp:
  - 78084 Mobile (Type 1)
  - 6534 Top Fixed (Type 2)
  - 6480 Bottom Fixed (Type 3)
